## Importing Libraries

In [54]:
import re
from collections import Counter

In [55]:
def load_text(filepath):
    with open(filepath, "r") as f:
        text = f.read()
    return text

In [56]:
def clean_text(text, lowercase=True):
    text = re.sub(r"\s+", " ", text)
    if lowercase:
        text = text.lower()
    return text.strip()

## Basic statistics

In [57]:
def basic_statistics(text):
    words = text.split()
    word_count = len(words)
    unique_words = len(set(words))

    freq = Counter(words)
    top20 = freq.most_common(20)

    print("===== BASIC STATISTICS =====")
    print("Total words:", word_count)
    print("Unique words:", unique_words)
    print("Top 20 most frequent words:")
    for w, c in top20:
        print(f"{w}: {c}")
    print()

In [1]:
raw = load_text('./datasets/science.txt')
raw += load_text('./datasets/rural.txt')
raw = clean_text(raw)

# Split train/test
split = int(0.9 * len(raw))
train_text = raw[:split]
test_text = raw[split:]

# Part 1: Stats
basic_statistics(train_text)

NameError: name 'load_text' is not defined

## Word-level BPE

In [59]:
from collections import defaultdict

EOW = "<end_of_word>"

def build_word_vocab(text: str):
    # word frequency dictionary: {"t h e <end_of_word>": count, ...}
    words = text.split()
    wf = Counter(words)
    vocab = {}
    for w, c in wf.items():
        symbols = " ".join(list(w)) + f" {EOW}"
        vocab[symbols] = c
    return vocab

def get_pair_freqs_wordlevel(vocab: dict) -> Counter:
    # weighted pair counts
    pair_freq = Counter()
    for tokenized_word, freq in vocab.items():
        symbols = tokenized_word.split()
        for i in range(len(symbols) - 1):
            pair_freq[(symbols[i], symbols[i+1])] += freq
    return pair_freq

def merge_pair_in_vocab(vocab: dict, pair: tuple[str, str]) -> dict:
    a, b = pair
    pattern = re.compile(rf"(?<!\S){re.escape(a)} {re.escape(b)}(?!\S)")
    replacement = a + b
    new_vocab = {}
    for tokenized_word, freq in vocab.items():
        new_word = pattern.sub(replacement, tokenized_word)
        new_vocab[new_word] = freq
    return new_vocab

def learn_bpe_wordlevel(text: str, K: int):
    vocab = build_word_vocab(text)
    merges = []
    for step in range(K):
        pair_freq = get_pair_freqs_wordlevel(vocab)
        if not pair_freq:
            break
        best_pair, best_count = pair_freq.most_common(1)[0]
        if best_count < 2:
            break
        vocab = merge_pair_in_vocab(vocab, best_pair)
        merges.append(best_pair)
        if (step + 1) % 100 == 0:
            print(f"[word-BPE] merge {step+1}/{K}: {best_pair} (count={best_count})")
    return merges

def apply_bpe_wordlevel_to_word(word: str, merges: list[tuple[str,str]]):
    # start from char-level + EOW
    symbols = list(word) + [EOW]
    # apply merges sequentially
    for a, b in merges:
        i = 0
        out = []
        while i < len(symbols):
            if i < len(symbols)-1 and symbols[i] == a and symbols[i+1] == b:
                out.append(a+b)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        symbols = out
    return symbols

def tokenize_wordlevel(text: str, merges: list[tuple[str,str]]):
    tokens = []
    for w in text.split():
        toks = apply_bpe_wordlevel_to_word(w, merges)
        tokens.extend(toks)
    return tokens

In [60]:
word_merges = learn_bpe_wordlevel(train_text, K=2000)

[word-BPE] merge 100/2000: ('co', 'm') (count=4677)
[word-BPE] merge 200/2000: ('mor', 'e<end_of_word>') (count=1985)
[word-BPE] merge 300/2000: ('b', 'r') (count=1217)
[word-BPE] merge 400/2000: ('australi', 'a') (count=877)
[word-BPE] merge 500/2000: ('th', 'in') (count=663)
[word-BPE] merge 600/2000: ('f', 'oo') (count=540)
[word-BPE] merge 700/2000: ('c', 'ell') (count=445)
[word-BPE] merge 800/2000: ('in', 'st') (count=368)
[word-BPE] merge 900/2000: ('b', 'an') (count=324)
[word-BPE] merge 1000/2000: ('v', 'al') (count=289)
[word-BPE] merge 1100/2000: ('h', 'er<end_of_word>') (count=257)
[word-BPE] merge 1200/2000: ('s', 'il') (count=232)
[word-BPE] merge 1300/2000: ('es', ',"<end_of_word>') (count=212)
[word-BPE] merge 1400/2000: ('tran', 's') (count=195)
[word-BPE] merge 1500/2000: ('cro', 'p<end_of_word>') (count=176)
[word-BPE] merge 1600/2000: ('institut', 'e<end_of_word>') (count=163)
[word-BPE] merge 1700/2000: ('me', 'et') (count=150)
[word-BPE] merge 1800/2000: ('ag', 'e

In [61]:
word_tokens = tokenize_wordlevel(test_text, word_merges)

In [18]:
import random
from IPython.display import display, HTML

def random_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

def colorize_tokens(tokens):
    html = ""
    for token in tokens:
        color = random_color()
        html += f'<span style="background-color:{color}; padding:2px; margin:1px; border-radius:4px;">{token}</span>'
    return HTML(html)

In [62]:
display(colorize_tokens(word_tokens[:2000]))

## Byte-level BPE

In [26]:
def get_pair_freqs_sequence(seq: list[str]) -> Counter:
    pair_freq = Counter()
    for i in range(len(seq) - 1):
        pair_freq[(seq[i], seq[i+1])] += 1
    return pair_freq

def merge_pair_in_sequence(seq: list[str], pair: tuple[str,str]) -> list[str]:
    a, b = pair
    merged = a + b
    out = []
    i = 0
    while i < len(seq):
        if i < len(seq)-1 and seq[i] == a and seq[i+1] == b:
            out.append(merged)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return out

def learn_bpe_bytelevel_char(text: str, K: int):
    # initial symbols: every character (including spaces)
    seq = list(text)
    merges = []
    for step in range(K):
        pair_freq = get_pair_freqs_sequence(seq)
        if not pair_freq:
            break
        best_pair, best_count = pair_freq.most_common(1)[0]
        if best_count < 2:
            break
        seq = merge_pair_in_sequence(seq, best_pair)
        merges.append(best_pair)
        if (step + 1) % 100 == 0:
            print(f"[byte/char-BPE] merge {step+1}/{K}: {best_pair} (count={best_count})")
    return merges

def tokenize_bytelevel_char(text: str, merges: list[tuple[str,str]]):
    seq = list(text)
    for a, b in merges:
        seq = merge_pair_in_sequence(seq, (a,b))
    return seq

In [41]:
byte_merges = learn_bpe_bytelevel_char(train_text, K=500)

[byte/char-BPE] merge 100/500: (' ', 're') (count=2368)
[byte/char-BPE] merge 200/500: ('d', 'u') (count=1072)
[byte/char-BPE] merge 300/500: ('. "', 'th') (count=678)
[byte/char-BPE] merge 400/500: (' a', 'd') (count=482)
[byte/char-BPE] merge 500/500: ('an', 's') (count=366)


In [42]:
byte_tokens = tokenize_bytelevel_char(test_text, byte_merges)

In [45]:
display(colorize_tokens(byte_tokens[:500]))

## Comparison metrics

In [38]:
def tokens_per_1000_chars(tokens: list[str], original_text: str) -> float:
    n_chars = len(original_text)
    return len(tokens) / (n_chars / 1000) if n_chars else 0.0

def tokens_per_word(tokens: list[str], original_text: str) -> float:
    n_words = len(original_text.split())
    return len(tokens) / n_words if n_words else 0.0

def cross_word_token_ratio(tokens: list[str]) -> float:
    # token contains a space (i.e., spans a word boundary)
    if not tokens:
        return 0.0
    return sum(1 for t in tokens if " " in t) / len(tokens)

def show_segmentation_on_words(test_text: str, word_merges, byte_merges, n=15):
    words = test_text.split()
    sample = words[:n]
    print("\nSample word segmentations:")
    for w in sample:
        w_tokens = apply_bpe_wordlevel_to_word(w, word_merges)
        # For byte-level, we need segmentation “inside the whole text”, but we can still show
        # how the word appears when tokenizing the *word with a leading space* which is typical in LLM tokenizers.
        byte_tokens = tokenize_bytelevel_char(" " + w, byte_merges)
        print(f"- {w}")
        print(f"  word-BPE: {w_tokens}")
        print(f"  byte-BPE: {byte_tokens}")

In [46]:
print("\nEVAL METRICS on TEST")
print("word-BPE tokens/1000 chars:", tokens_per_1000_chars(word_tokens, test_text))
print("word-BPE tokens/word:", tokens_per_word(word_tokens, test_text))
print("word-BPE cross-word ratio:", cross_word_token_ratio(word_tokens))  # should be ~0

print("byte-BPE tokens/1000 chars:", tokens_per_1000_chars(byte_tokens, test_text))
print("byte-BPE tokens/word:", tokens_per_word(byte_tokens, test_text))
print("byte-BPE cross-word ratio:", cross_word_token_ratio(byte_tokens))  # should be >0

# 5) qualitative
show_segmentation_on_words(test_text, word_merges, byte_merges, n=20)


EVAL METRICS on TEST
word-BPE tokens/1000 chars: 282.7045169638885
word-BPE tokens/word: 1.7455680399500624
word-BPE cross-word ratio: 0.0
byte-BPE tokens/1000 chars: 406.5806677719816
byte-BPE tokens/word: 2.5104452767374115
byte-BPE cross-word ratio: 0.3455337112798241

Sample word segmentations:
- ing
  word-BPE: ['ing<end_of_word>']
  byte-BPE: [' ', 'ing']
- to
  word-BPE: ['to<end_of_word>']
  byte-BPE: [' to']
- assess
  word-BPE: ['ass', 'ess<end_of_word>']
  byte-BPE: [' as', 's', 'ess']
- respondents'
  word-BPE: ['respon', 'd', 'ent', "s'<end_of_word>"]
  byte-BPE: [' re', 'sp', 'on', 'd', 'ent', 's', "'"]
- mental
  word-BPE: ['ment', 'al<end_of_word>']
  byte-BPE: [' m', 'ent', 'al']
- health,
  word-BPE: ['health', ',<end_of_word>']
  byte-BPE: [' h', 'eal', 'th', ',']
- but
  word-BPE: ['but<end_of_word>']
  byte-BPE: [' b', 'ut']
- says
  word-BPE: ['says<end_of_word>']
  byte-BPE: [' says']
- it
  word-BPE: ['it<end_of_word>']
  byte-BPE: [' ', 'it']
- does
  word-BPE